# IEP-2001.3 — 하이브리드 방법E cosine 컬렉션 재실험

| 항목 | 내용 |
| :--- | :--- |
| **목적** | IEP-2001.1(방법E)을 L2 컬렉션이 아닌 cosine 컬렉션 기준으로 재실험, 배포 환경과 측정 조건 통일 |
| **Baseline** | IEP-1001 Solar v2 (Recall 0.6549, Precision 0.5608, Faith보수 0.1424, AR 0.4075) |
| **결과** | Recall 0.7609 / Precision 0.6125 / Faith보수 0.2017 / AR 0.5112 / 거절 0/100 |
| **실행 환경** | Google Colab T4 |
| **실행일** | 2026-05-08 |

## IEP-2001.1 대비 변경점

| 항목 | IEP-2001.1 | IEP-2001.3 (이번) |
| :--- | :--- | :--- |
| ChromaDB 컬렉션 | `ipcc_1001_case3_v1` (L2) | `ipcc_1001_case3_cosine_v1` (cosine) |
| 벡터 검색 | `similarity_search_with_relevance_scores` | `similarity_search_with_score` + `1-distance` 직접 변환 |
| TOP_K | 3 | 10 (배포 설정과 동일) |
| Baseline | IEP-1001 Solar v1 (judge 혼용) | IEP-1001 Solar v2 (전 지표 Solar) |

## Cell 1 — 패키지 설치

In [ ]:
import subprocess
subprocess.run(["pip", "uninstall", "numpy", "-y"], capture_output=True)
subprocess.run(["pip", "install", "numpy==1.26.4", "--force-reinstall", "-q"], capture_output=True)

!pip install rank-bm25 kiwipiepy --quiet
!pip install chromadb==0.5.11 langchain-chroma==0.1.4 langchain-community langchain-huggingface \
             sentence-transformers openai python-dotenv ragas --quiet

import numpy as np
print(f"numpy: {np.__version__}")

## Cell 2 — Google Drive 마운트 + 경로 설정

> ⚠️ 본인 Drive 경로로 수정하세요

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, getpass

# ── ⚠️ 본인 경로로 수정 ──────────────────────────────────
BASE_DIR    = "/content/drive/MyDrive/IPCC_data"
CHROMA_DIR  = f"{BASE_DIR}/chroma_cosine"           # cosine 컬렉션 경로
GOLDEN_PATH = f"{BASE_DIR}/IEP_1002/golden_dataset_100.csv"
RESULTS_DIR = f"{BASE_DIR}/IEP_2001/results_method_e_cosine"
# ─────────────────────────────────────────────────────────

COLLECTION_NAME = "ipcc_1001_case3_cosine_v1"
EMBED_MODEL     = "jhgan/ko-sroberta-multitask"
SOLAR_BASE_URL  = "https://api.upstage.ai/v1"
SOLAR_MODEL     = "solar-pro3"

os.makedirs(RESULTS_DIR, exist_ok=True)

# API 키 — Colab Secrets 사용 권장
# from google.colab import userdata
# UPSTAGE_API_KEY = userdata.get('UPSTAGE_API_KEY')
UPSTAGE_API_KEY = getpass.getpass("UPSTAGE_API_KEY: ")
os.environ["UPSTAGE_API_KEY"] = UPSTAGE_API_KEY

print(f"ChromaDB: {os.path.exists(CHROMA_DIR)}")
print(f"골든 데이터셋: {os.path.exists(GOLDEN_PATH)}")
print(f"컬렉션: {COLLECTION_NAME}")

## Cell 3 — ChromaDB(cosine) 로드

- `similarity_search_with_score` + `1 - distance` 직접 변환 (LangChain 버전 의존성 제거)

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR
)

# 전체 청크 추출 (BM25 인덱스 구축용)
raw = vectorstore._collection.get(include=["documents", "metadatas"])
docs_text = raw["documents"]
docs_meta = raw["metadatas"]

print(f"청크 수: {vectorstore._collection.count()}")

# cosine score 범위 확인
test_results = vectorstore.similarity_search_with_score("지구 온난화가 해수면에 미치는 영향은?", k=3)
for doc, dist in test_results:
    print(f"  distance={dist:.4f} → similarity={1-dist:.4f}")

## Cell 4 — BM25 인덱스 구축

In [ ]:
from rank_bm25 import BM25Okapi
import time

try:
    from kiwipiepy import Kiwi
    kiwi = Kiwi()
    def tokenize(text):
        return [t.form for t in kiwi.tokenize(text)]
    print("토크나이저: kiwipiepy")
except Exception:
    def tokenize(text):
        return text.split()
    print("토크나이저: 공백 split (fallback)")

t0 = time.time()
tokenized_corpus = [tokenize(doc) for doc in docs_text]
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 인덱스 완료: {time.time()-t0:.1f}초")

## Cell 5 — 검색 함수 정의 (방법E)

- 벡터 + BM25 → RRF → TOP_K=10
- TOP_K 중 최고 cosine similarity < THRESHOLD(0.20) → 거절

In [ ]:
from langchain_core.documents import Document

TOP_K                = 10
BM25_CANDIDATE_K     = 10
SIMILARITY_THRESHOLD = 0.20
RRF_K                = 60

def vector_search(question, k=BM25_CANDIDATE_K):
    """cosine 컬렉션 벡터 검색 — 1-distance 직접 변환"""
    raw = vectorstore.similarity_search_with_score(question, k=k)
    return [(doc, 1 - dist) for doc, dist in raw]

def bm25_search(question, k=BM25_CANDIDATE_K):
    """BM25 검색"""
    tokens = tokenize(question)
    scores = bm25.get_scores(tokens)
    top_idx = scores.argsort()[::-1][:k]
    return [(Document(page_content=docs_text[i], metadata=docs_meta[i]), float(scores[i])) for i in top_idx]

def rrf_fusion(v_results, b_results, top_k=TOP_K):
    """Reciprocal Rank Fusion"""
    rrf_scores, doc_map, vec_scores = {}, {}, {}

    def key(doc):
        return doc.page_content[:80]

    for rank, (doc, score) in enumerate(v_results):
        k_ = key(doc)
        rrf_scores[k_] = rrf_scores.get(k_, 0.0) + 1.0 / (RRF_K + rank + 1)
        doc_map[k_] = doc
        vec_scores[k_] = score

    for rank, (doc, _) in enumerate(b_results):
        k_ = key(doc)
        rrf_scores[k_] = rrf_scores.get(k_, 0.0) + 1.0 / (RRF_K + rank + 1)
        if k_ not in doc_map:
            doc_map[k_] = doc

    top_keys = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]
    return [(doc_map[k_], rrf_scores[k_], vec_scores.get(k_, 0.0)) for k_ in top_keys]

def hybrid_search(question):
    """방법E: 하이브리드 검색 + cosine threshold 거절"""
    top = rrf_fusion(vector_search(question), bm25_search(question))
    if not top:
        return []
    best_vec = max(vs for _, _, vs in top)
    if best_vec < SIMILARITY_THRESHOLD:
        return []
    return [(doc, rrf_s) for doc, rrf_s, _ in top]

print(f"TOP_K={TOP_K}, THRESHOLD={SIMILARITY_THRESHOLD}")

## Cell 6 — 스모크 테스트

In [ ]:
test_cases = [
    ("1.5°C 목표를 달성하기 위해 필요한 탄소 감축량은?", "관련"),
    ("기후변화가 생태계에 미치는 영향은?",              "관련"),
    ("환경 보호를 위해 개인이 할 수 있는 일은?",        "경계"),
    ("오늘 점심 메뉴 추천해줘",                        "범위 밖"),
]

for q, label in test_cases:
    v = vector_search(q, k=3)
    best = max(s for _, s in v)
    result = hybrid_search(q)
    verdict = "✅ 통과" if result else "❌ 거절"
    print(f"[{label}] {verdict} | 최고 similarity={best:.4f} | {q[:30]}")

## Cell 7 — 골든 데이터셋 100개 검색

In [ ]:
import csv, time

with open(GOLDEN_PATH, "r", encoding="utf-8") as f:
    golden = list(csv.DictReader(f))
print(f"골든 데이터셋: {len(golden)}개")

hybrid_dataset, rejected_count = [], 0
t0 = time.time()

for i, item in enumerate(golden):
    results = hybrid_search(item["user_input"])
    contexts = [doc.page_content for doc, _ in results]
    if not contexts:
        rejected_count += 1
    hybrid_dataset.append({
        "question":     item["user_input"],
        "ground_truth": item["reference"],
        "contexts":     contexts,
        "rejected":     len(contexts) == 0,
    })
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/100 완료")

print(f"검색 완료: {time.time()-t0:.0f}초 | 거절: {rejected_count}/100")

## Cell 8 — Solar 답변 생성

In [ ]:
from openai import OpenAI

solar = OpenAI(api_key=UPSTAGE_API_KEY, base_url=SOLAR_BASE_URL)

RAG_PROMPT = """당신은 IPCC AR6 기후변화 보고서 전문 안내 AI입니다.
반드시 아래 제공된 문서 내용만을 근거로 답변하세요.
문서에 없는 내용은 "해당 내용은 IPCC 보고서에서 찾을 수 없습니다."라고 답하세요.

[참고 문서]
{context}

[질문]
{question}

[답변]"""

REJECTION = "죄송합니다. 해당 질문은 IPCC AR6 보고서의 범위를 벗어납니다."

def generate(question, contexts):
    if not contexts:
        return REJECTION
    ctx = "\n\n".join(f"[청크 {i+1}]\n{c}" for i, c in enumerate(contexts))
    for attempt in range(2):
        try:
            resp = solar.chat.completions.create(
                model=SOLAR_MODEL,
                messages=[{"role": "user", "content": RAG_PROMPT.format(context=ctx, question=question)}],
                max_tokens=800,
                timeout=30
            )
            answer = resp.choices[0].message.content
            if "</think>" in answer:
                answer = answer.split("</think>")[-1].strip()
            return answer
        except Exception as e:
            if attempt == 0:
                print(f"재시도... ({e})")
    return "답변 생성 실패"

t0 = time.time()
for i, item in enumerate(hybrid_dataset):
    item["answer"] = generate(item["question"], item["contexts"])
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/100 완료 ({time.time()-t0:.0f}초)")
print(f"답변 생성 완료: {time.time()-t0:.0f}초")

## Cell 9 — RAGAS 설정

Solar judge + jhgan 임베딩 (IEP-1001 Solar v2와 동일 조건)

In [ ]:
import pandas as pd
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from ragas import EvaluationDataset

# IEP-1001 Solar v2 baseline
IEP1001 = {
    "전체":      {"context_recall": 0.6549, "context_precision": 0.5608, "faithfulness": 0.3165, "answer_relevancy": 0.4075},
    "사실 확인": {"context_recall": 0.8000, "context_precision": 0.6006, "faithfulness": 0.5646, "answer_relevancy": 0.6191},
    "비교":      {"context_recall": 0.4600, "context_precision": 0.6220, "faithfulness": 0.1429, "answer_relevancy": 0.4089},
    "의견/예측": {"context_recall": 0.7014, "context_precision": 0.7623, "faithfulness": 0.3250, "answer_relevancy": 0.4867},
    "범위 밖":   {"context_recall": 0.6600, "context_precision": 0.2598, "faithfulness": 0.1520, "answer_relevancy": 0.1155},
}

ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(
        model=SOLAR_MODEL,
        openai_api_key=UPSTAGE_API_KEY,
        openai_api_base=SOLAR_BASE_URL,
        temperature=0,
        max_tokens=512,  # ⚠️ NaN 감소를 위해 1024 권장
    )
)
ragas_emb   = LangchainEmbeddingsWrapper(embeddings)
run_config  = RunConfig(max_workers=2, timeout=120, max_retries=3)

eval_records = [{
    "id":                 golden[i].get("ID", ""),
    "type":               golden[i].get("Type", ""),
    "user_input":         h["question"],
    "retrieved_contexts": h["contexts"],
    "response":           h["answer"],
    "reference":          h["ground_truth"],
} for i, h in enumerate(hybrid_dataset)]

eval_dataset = EvaluationDataset.from_pandas(pd.DataFrame([{
    "user_input":         r["user_input"],
    "retrieved_contexts": r["retrieved_contexts"],
    "response":           r["response"],
    "reference":          r["reference"],
} for r in eval_records]))

print(f"EvaluationDataset: {len(eval_dataset)}개")

## Cell 10 — Context Recall + Precision

Baseline: Recall 0.6549 / Precision 0.5608

In [ ]:
from ragas import evaluate
from ragas.metrics import ContextRecall, ContextPrecision

result_ret = evaluate(
    dataset=eval_dataset,
    metrics=[ContextRecall(), ContextPrecision()],
    llm=ragas_llm,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)

df_retrieval = result_ret.to_pandas()
df_retrieval[["id", "type"]] = pd.DataFrame([[r["id"], r["type"]] for r in eval_records])
df_retrieval.to_csv(f"{RESULTS_DIR}/retrieval.csv", index=False, encoding="utf-8-sig")

r_mean = df_retrieval["context_recall"].mean(skipna=True)
p_mean = df_retrieval["context_precision"].mean(skipna=True)
print(f"Context Recall    : {r_mean:.4f}  (diff: {r_mean-0.6549:+.4f})")
print(f"Context Precision : {p_mean:.4f}  (diff: {p_mean-0.5608:+.4f})")

## Cell 11 — Faithfulness

Baseline: 0.3165 (보수적 0.1424)

In [ ]:
from ragas.metrics import Faithfulness

result_faith = evaluate(
    dataset=eval_dataset,
    metrics=[Faithfulness()],
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)

df_faith = result_faith.to_pandas()
df_faith[["id", "type"]] = pd.DataFrame([[r["id"], r["type"]] for r in eval_records])
df_faith.to_csv(f"{RESULTS_DIR}/faithfulness.csv", index=False, encoding="utf-8-sig")

f_mean = df_faith["faithfulness"].mean(skipna=True)
f_nan  = df_faith["faithfulness"].isna().sum()
f_cons = round(f_mean * (100 - f_nan) / 100, 4)
print(f"Faithfulness (낙관): {f_mean:.4f}  (NaN: {f_nan}개)")
print(f"Faithfulness (보수): {f_cons:.4f}  (diff: {f_cons-0.1424:+.4f})")

## Cell 12 — Answer Relevancy

Baseline: 0.4075 / strictness=1 (Solar n=1 제약)

In [ ]:
from ragas.metrics import AnswerRelevancy

result_ar = evaluate(
    dataset=eval_dataset,
    metrics=[AnswerRelevancy(strictness=1)],
    llm=ragas_llm,
    embeddings=ragas_emb,
    run_config=run_config,
    batch_size=1,
    raise_exceptions=False,
)

df_ar = result_ar.to_pandas()
df_ar[["id", "type"]] = pd.DataFrame([[r["id"], r["type"]] for r in eval_records])
df_ar.to_csv(f"{RESULTS_DIR}/relevancy.csv", index=False, encoding="utf-8-sig")

ar_mean = df_ar["answer_relevancy"].mean(skipna=True)
print(f"Answer Relevancy  : {ar_mean:.4f}  (diff: {ar_mean-0.4075:+.4f})")

## Cell 13 — 결과 정리 + 비교표

In [ ]:
import numpy as np

TYPES       = ["사실 확인", "비교", "의견/예측", "범위 밖"]
ALL_METRICS = ["context_recall", "context_precision", "faithfulness", "answer_relevancy"]

df_raw = pd.DataFrame({
    "id":                [r["id"]   for r in eval_records],
    "type":              [r["type"] for r in eval_records],
    "context_recall":    df_retrieval["context_recall"].values,
    "context_precision": df_retrieval["context_precision"].values,
    "faithfulness":      df_faith["faithfulness"].values,
    "answer_relevancy":  df_ar["answer_relevancy"].values,
})

overall  = df_raw[ALL_METRICS].mean(skipna=True).round(4)
nan_cnt  = df_raw[ALL_METRICS].isna().sum()
summary  = df_raw.groupby("type")[ALL_METRICS].mean().round(4)
f_cons   = round(overall["faithfulness"] * (100 - nan_cnt["faithfulness"]) / 100, 4)

print("=" * 65)
print("  방법E cosine (IEP-2001.3) vs IEP-1001 Solar v2")
print("=" * 65)
print(pd.DataFrame([
    {"단계": "IEP-1001 벡터 단독", "Recall": 0.6549, "Precision": 0.5608, "Faith(보수)": 0.1424, "AR": 0.4075, "거절": "-"},
    {"단계": "방법E cosine",       "Recall": overall["context_recall"], "Precision": overall["context_precision"],
     "Faith(보수)": f_cons, "AR": overall["answer_relevancy"], "거절": f"{rejected_count}/100"},
]).to_string(index=False))

print("\n[유형별 상세]")
for m in ALL_METRICS:
    print(f"\n  {m}")
    for t in TYPES:
        v = summary.loc[t, m] if t in summary.index else float("nan")
        b = IEP1001.get(t, {}).get(m, float("nan"))
        diff = f"{v-b:+.4f}" if not (np.isnan(v) or np.isnan(b)) else "N/A"
        print(f"    {t:<10} {v:.4f}  baseline={b:.4f}  diff={diff}")
    print(f"    전체       {overall[m]:.4f}  baseline={IEP1001['전체'][m]:.4f}  diff={overall[m]-IEP1001['전체'][m]:+.4f}")

print(f"\n[생존 편향 보정]")
print(f"  Faithfulness: 낙관={overall['faithfulness']:.4f} / 유효={100-nan_cnt['faithfulness']}개 / 보수={f_cons:.4f}")

print("\n[README 마크다운]")
print(f"| 벡터 단독 thr=0.40 (배포) | 0.6549 | 0.5608 | 0.1424 | 0.4075 | — |")
print(f"| 하이브리드 방법E thr=0.20 (cosine) | {overall['context_recall']:.4f} | {overall['context_precision']:.4f} | {f_cons:.4f} | {overall['answer_relevancy']:.4f} | {rejected_count}/100 |")

## Cell 14 — Drive 저장

In [ ]:
import json
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M")

df_raw.to_csv(f"{RESULTS_DIR}/iep2001_3_raw_{ts}.csv", index=False, encoding="utf-8-sig")

df_s = summary.copy()
df_s.loc["전체"] = overall
df_s.to_csv(f"{RESULTS_DIR}/iep2001_3_summary_{ts}.csv", encoding="utf-8-sig")

with open(f"{RESULTS_DIR}/iep2001_3_dataset_{ts}.json", "w", encoding="utf-8") as f:
    json.dump(hybrid_dataset, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {RESULTS_DIR}")
print(f"Faithfulness NaN: {nan_cnt['faithfulness']}개 — 다음 실험부터 max_tokens=1024 권장")